# ODI Migration: Dept_pkg Conversion
**Source File:** Dept_MAp.sql
**Target Workspace:** workspace.odi_trg
**Conversion Date:** 2024-05-22

In [ ]:
dbutils.widgets.text("v_lst_dt", "")
dbutils.widgets.text("ODI_SESS_NO", "45a8cc52-cb60-43c0-9b43-3c2c176192e9")
dbutils.widgets.text("DATASOURCE_NUM_ID", "1")

## ETL Parameters

In [ ]:
%sql
-- SCEN_TASK_NO in {1}: Initialize last run date
CREATE OR REPLACE TEMPORARY VIEW v_lst_dt_init AS
SELECT 
    COALESCE(
        (SELECT date_format(last_run_dt, 'dd-MM-yy') 
         FROM workspace.odi_trg.log_table1 
         WHERE pkg_name = 'Dept_pkg'),
        date_format(current_timestamp(), 'dd-MM-yy')
    ) AS etl_last_run_dt;

In [ ]:
%sql
-- SCEN_TASK_NO in {2}: Update log table for current run
UPDATE workspace.odi_trg.log_table1 
SET last_run_dt = current_timestamp() 
WHERE pkg_name = 'Dept_pkg';

## Staging Table: C$_0FILTER

In [ ]:
%sql
-- SCEN_TASK_NO in {30}
DROP TABLE IF EXISTS workspace.odi_trg.c_0filter;

In [ ]:
%sql
-- SCEN_TASK_NO in {40}
CREATE TABLE workspace.odi_trg.c_0filter
(
    DEPARTMENT_ID   BIGINT,
    DEPARTMENT_NAME STRING,
    MANAGER_ID      BIGINT,
    LOCATION_ID     BIGINT,
    LAST_UPD_DT     TIMESTAMP
)
USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO in {50}: Load from source to staging
INSERT INTO workspace.odi_trg.c_0filter
(
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
)
SELECT 
    SRC_DEPARTMENTS_1.DEPARTMENT_ID,
    SRC_DEPARTMENTS_1.DEPARTMENT_NAME,
    SRC_DEPARTMENTS_1.MANAGER_ID,
    SRC_DEPARTMENTS_1.LOCATION_ID,
    SRC_DEPARTMENTS_1.LAST_UPD_DT
FROM workspace.odi_src.src_departments AS SRC_DEPARTMENTS_1
WHERE SRC_DEPARTMENTS_1.LAST_UPD_DT >= to_date('${v_lst_dt}', 'dd-MM-yy');

In [ ]:
%sql
SELECT COUNT(*) AS c_staging_count FROM workspace.odi_trg.c_0filter;

In [ ]:
%sql
-- SCEN_TASK_NO in {60}: Gather stats
OPTIMIZE workspace.odi_trg.c_0filter;

## Flow Table: I$_TRG_DEPARTMENTS

In [ ]:
%sql
-- SCEN_TASK_NO in {80}
DROP TABLE IF EXISTS workspace.odi_trg.i_trg_departments_flow;

In [ ]:
%sql
-- SCEN_TASK_NO in {90}
CREATE TABLE workspace.odi_trg.i_trg_departments_flow
(
    DEPARTMENT_ID       BIGINT,
    DEPARTMENT_NAME     STRING,
    MANAGER_ID          BIGINT,
    LOCATION_ID         BIGINT,
    LAST_UPD_DT         TIMESTAMP,
    IND_UPDATE          STRING,
    rowid_spark         STRING
)
USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO in {100}: Detection logic
INSERT INTO workspace.odi_trg.i_trg_departments_flow
(
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT,
    IND_UPDATE,
    rowid_spark
)
SELECT 
    S.DEPARTMENT_ID,
    S.DEPARTMENT_NAME,
    S.MANAGER_ID,
    S.LOCATION_ID,
    S.LAST_UPD_DT,
    'I' AS IND_UPDATE,
    CAST(monotonically_increasing_id() AS STRING) AS rowid_spark
FROM (
    SELECT 
        FILTER_A.DEPARTMENT_ID,
        FILTER_A.DEPARTMENT_NAME,
        FILTER_A.MANAGER_ID,
        FILTER_A.LOCATION_ID,
        FILTER_A.LAST_UPD_DT
    FROM workspace.odi_trg.c_0filter AS FILTER_A
) AS S
WHERE NOT EXISTS (
    SELECT 1 FROM workspace.odi_trg.trg_departments AS T
    WHERE T.DEPARTMENT_ID = S.DEPARTMENT_ID 
    AND (T.DEPARTMENT_NAME = S.DEPARTMENT_NAME OR (T.DEPARTMENT_NAME IS NULL AND S.DEPARTMENT_NAME IS NULL))
    AND (T.MANAGER_ID = S.MANAGER_ID OR (T.MANAGER_ID IS NULL AND S.MANAGER_ID IS NULL))
    AND (T.LOCATION_ID = S.LOCATION_ID OR (T.LOCATION_ID IS NULL AND S.LOCATION_ID IS NULL))
    AND (T.LAST_UPD_DT = S.LAST_UPD_DT OR (T.LAST_UPD_DT IS NULL AND S.LAST_UPD_DT IS NULL))
);

In [ ]:
%sql
-- SCEN_TASK_NO in {110} & {120} & {170}
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.odi_trg.i_trg_departments_flow ZORDER BY (DEPARTMENT_ID);

## Error and Audit Handling

In [ ]:
%sql
-- SCEN_TASK_NO in {130}
CREATE TABLE IF NOT EXISTS workspace.odi_trg.snp_check_tab
(
    CATALOG_NAME    STRING,
    SCHEMA_NAME     STRING,
    RESOURCE_NAME   STRING,
    FULL_RES_NAME   STRING,
    ERR_TYPE        STRING,
    ERR_MESS        STRING,
    CHECK_DATE      TIMESTAMP,
    ORIGIN          STRING,
    CONS_NAME       STRING,
    CONS_TYPE       STRING,
    ERR_COUNT       BIGINT
) USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO in {140}
DELETE FROM workspace.odi_trg.snp_check_tab
WHERE SCHEMA_NAME = 'ODI_TRG'
AND ORIGIN = '(7)test.Dept_MAp'
AND ERR_TYPE = 'F';

In [ ]:
%sql
-- SCEN_TASK_NO in {150}
CREATE TABLE IF NOT EXISTS workspace.odi_trg.e_trg_departments
(
    ODI_ROW_ID      STRING,
    ODI_ERR_TYPE    STRING,
    ODI_ERR_MESS    STRING,
    ODI_CHECK_DATE  TIMESTAMP,
    DEPARTMENT_ID   BIGINT,
    DEPARTMENT_NAME STRING,
    MANAGER_ID      BIGINT,
    LOCATION_ID     BIGINT,
    LAST_UPD_DT     TIMESTAMP,
    ODI_ORIGIN      STRING,
    ODI_CONS_NAME   STRING,
    ODI_CONS_TYPE   STRING,
    ODI_PK          STRING,
    ODI_SESS_NO     STRING
) USING DELTA;

In [ ]:
%sql
-- SCEN_TASK_NO in {160}
DELETE FROM workspace.odi_trg.e_trg_departments
WHERE (ODI_ERR_TYPE = 'S' AND 1 = 0)
OR (ODI_ERR_TYPE = 'F' AND ODI_ORIGIN = '(7)test.Dept_MAp');

In [ ]:
%sql
-- SCEN_TASK_NO in {180}: PK Violation Detection
INSERT INTO workspace.odi_trg.e_trg_departments
(
    ODI_PK,
    ODI_SESS_NO,
    ODI_ROW_ID,
    ODI_ERR_TYPE,
    ODI_ERR_MESS,
    ODI_ORIGIN,
    ODI_CHECK_DATE,
    ODI_CONS_NAME,
    ODI_CONS_TYPE,
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
)
SELECT 
    uuid(),
    '${ODI_SESS_NO}',
    TRG_DEPARTMENTS.rowid_spark,
    'F',
    'ODI-15064: The primary key PK_department_id is not unique.',
    '(7)test.Dept_MAp',
    current_timestamp(),
    'PK_department_id',
    'PK',
    TRG_DEPARTMENTS.DEPARTMENT_ID,
    TRG_DEPARTMENTS.DEPARTMENT_NAME,
    TRG_DEPARTMENTS.MANAGER_ID,
    TRG_DEPARTMENTS.LOCATION_ID,
    TRG_DEPARTMENTS.LAST_UPD_DT
FROM workspace.odi_trg.i_trg_departments_flow AS TRG_DEPARTMENTS
WHERE EXISTS (
    SELECT 1
    FROM workspace.odi_trg.i_trg_departments_flow AS SUB
    WHERE SUB.DEPARTMENT_ID = TRG_DEPARTMENTS.DEPARTMENT_ID
    GROUP BY SUB.DEPARTMENT_ID
    HAVING COUNT(1) > 1
);

In [ ]:
%sql
-- SCEN_TASK_NO in {190}: Not Null Violation Detection
INSERT INTO workspace.odi_trg.e_trg_departments
(
    ODI_PK,
    ODI_SESS_NO,
    ODI_ROW_ID,
    ODI_ERR_TYPE,
    ODI_ERR_MESS,
    ODI_CHECK_DATE,
    ODI_ORIGIN,
    ODI_CONS_NAME,
    ODI_CONS_TYPE,
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
)
SELECT 
    uuid(),
    '${ODI_SESS_NO}',
    rowid_spark,
    'F',
    'ODI-15066: The column DEPARTMENT_NAME cannot be null.',
    current_timestamp(),
    '(7)test.Dept_MAp',
    'DEPARTMENT_NAME',
    'NN',
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
FROM workspace.odi_trg.i_trg_departments_flow
WHERE DEPARTMENT_NAME IS NULL;

In [ ]:
%sql
-- SCEN_TASK_NO in {210}: Remove erroneous records from flow
MERGE INTO workspace.odi_trg.i_trg_departments_flow AS T
USING workspace.odi_trg.e_trg_departments AS S
ON S.ODI_SESS_NO = '${ODI_SESS_NO}'
AND T.rowid_spark = S.ODI_ROW_ID
WHEN MATCHED THEN DELETE;

In [ ]:
%sql
-- SCEN_TASK_NO in {220}: Audit Summary
INSERT INTO workspace.odi_trg.snp_check_tab
(
    SCHEMA_NAME,
    RESOURCE_NAME,
    FULL_RES_NAME,
    ERR_TYPE,
    ERR_MESS,
    CHECK_DATE,
    ORIGIN,
    CONS_NAME,
    CONS_TYPE,
    ERR_COUNT
)
SELECT 
    'ODI_TRG',
    'TRG_DEPARTMENTS',
    'ODI_TRG.TRG_DEPARTMENTS',
    E.ODI_ERR_TYPE,
    E.ODI_ERR_MESS,
    E.ODI_CHECK_DATE,
    E.ODI_ORIGIN,
    E.ODI_CONS_NAME,
    E.ODI_CONS_TYPE,
    COUNT(1)
FROM workspace.odi_trg.e_trg_departments AS E
WHERE E.ODI_ERR_TYPE = 'F'
AND E.ODI_ORIGIN = '(7)test.Dept_MAp'
GROUP BY E.ODI_ERR_TYPE, E.ODI_ERR_MESS, E.ODI_CHECK_DATE, E.ODI_ORIGIN, E.ODI_CONS_NAME, E.ODI_CONS_TYPE;

## Final Merge to Target

In [ ]:
%sql
-- SCEN_TASK_NO in {230}: Mark Records for Update (F.10 Rewrite)
MERGE INTO workspace.odi_trg.i_trg_departments_flow AS T
USING workspace.odi_trg.trg_departments AS S
ON T.DEPARTMENT_ID = S.DEPARTMENT_ID
WHEN MATCHED THEN UPDATE SET T.IND_UPDATE = 'U';

In [ ]:
%sql
-- SCEN_TASK_NO in {270} + {280}: Synchronize Target (Combined MERGE)
MERGE INTO workspace.odi_trg.trg_departments AS T
USING workspace.odi_trg.i_trg_departments_flow AS S
ON T.DEPARTMENT_ID = S.DEPARTMENT_ID
WHEN MATCHED AND S.IND_UPDATE = 'U' THEN UPDATE SET
    T.DEPARTMENT_NAME = S.DEPARTMENT_NAME,
    T.MANAGER_ID      = S.MANAGER_ID,
    T.LOCATION_ID     = S.LOCATION_ID,
    T.LAST_UPD_DT     = S.LAST_UPD_DT
WHEN NOT MATCHED AND S.IND_UPDATE = 'I' THEN INSERT
(
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
)
VALUES
(
    S.DEPARTMENT_ID,
    S.DEPARTMENT_NAME,
    S.MANAGER_ID,
    S.LOCATION_ID,
    S.LAST_UPD_DT
);

In [ ]:
%sql
-- Target Optimization
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.odi_trg.trg_departments ZORDER BY (DEPARTMENT_ID);

## Cleanup

In [ ]:
%sql
-- SCEN_TASK_NO in {340} & {360}
DROP TABLE IF EXISTS workspace.odi_trg.c_0filter;
DROP TABLE IF EXISTS workspace.odi_trg.i_trg_departments_flow;

## Validation

In [ ]:
%sql
SELECT COUNT(*) AS total_target_records FROM workspace.odi_trg.trg_departments;

In [ ]:
%sql
SELECT * FROM workspace.odi_trg.snp_check_tab WHERE ORIGIN = '(7)test.Dept_MAp' ORDER BY CHECK_DATE DESC LIMIT 10;